# Warehouse Delivery Robot: Search Algorithms

## How to run
1. Run the cells from top to bottom 
2. To change the map, edit the line `GRID = MAP_A` in the **Maps** cell and set it to `MAP_B`, `MAP_C`, or your own list of strings.
3. The **detailed output** cell prints the path, cost, nodes expanded and a drawing of the path for the map in `GRID`.
4. The **results table** cell runs every algorithm on MAP_A, MAP_B and MAP_C.

## Map symbols
| Symbol | Meaning |
|---|---|
| S / G | Start / Goal (entering G costs 1, S is not counted) |
| . | Normal floor (cost 1) |
| ~ | Rough floor (cost 3) |
| # | Wall (cannot enter) |

In [38]:
from collections import deque   # queue for BFS
import heapq                    # priority queue for UCS, Greedy, A*
import time                     # to measure run time

# The Maps 

In [39]:
MAP_A = [
    "S..#....",
    "##.#.##.",
    "....~~..",
    ".####.#.",
    "...~..#G"
]

MAP_B = [
    "S~~~~~~~G",
    ".#######.",
    ".........",
    ".#.#.#.#.",
    "........."
]

MAP_C = [
    "S..#....",
    ".#.#.##.",
    "...#..#.",
    ".###.##.",
    "....#..G"
]

COSTS = {      # cost of ENTERING each cell (#is a wall, so it is not here)
    ".": 1,
    "~": 3,
    "S": 0,
    "G": 1,
}

GRID = MAP_A   #  CHANGE THIS LINE to MAP_B, MAP_C or your own map

## 1. Problem Formulation

A delivery robot is working in a warehouse. It starts on the cell marked **S** and has to reach the cell marked **G**. It can only move one cell at a time, either up, down, left or right, and it can never walk through a wall. Some of the floor is smooth and some of it is rough, and the rough floor is harder to cross, so it costs more. The goal of this task is to find a good path for the robot from S to G using search algorithms.

To turn this into a search problem, we describe the robot's situation using its position on the map. A **state** is simply the cell the robot is standing on, written as a (row, column) pair. The **initial state** is the cell containing S, which the program finds by scanning the map. The **goal test** checks whether the robot's current cell is the cell containing G.

From any cell, the robot's **actions** are to move up, right, down or left. An action is only allowed if the new cell is still inside the map and is not a wall. Moving in an allowed direction simply takes the robot to the neighbouring cell.

Every move has a **cost** that depends on the cell the robot steps into. Stepping onto normal floor (.) costs 1, stepping onto rough floor (~) costs 3, and stepping onto G costs 1. The start cell S is never stepped into, so it adds nothing. The **cost of a path** is the sum of all its moves. For example, walking over three normal cells and then one rough cell costs 1 + 1 + 1 + 3 = 6. The best solution is the path from S to G with the lowest total cost, not necessarily the one with the fewest steps.

This distinction matters because a short path can end up being more expensive if it crosses a lot of rough floor. MAP_B was designed to show exactly this. Some maps may also have no solution at all, because walls can completely cut G off from S. In that case the correct answer is "No path found", and MAP_C was designed to test this.

Finally, the robot always checks its directions in the same order: up, right, down, then left. This order only matters when several paths are equally good, where it can change which one DFS and Greedy return. It never changes the cost found by UCS and A\*, which are guaranteed to find the cheapest path.

In [40]:
class GridProblem:
    def __init__(self, grid):
        # accept a list of strings (or one multi-line string)
        self.grid = grid if isinstance(grid, list) else [l.strip() for l in grid.strip().splitlines()]
        self.rows = len(self.grid)
        self.cols = len(self.grid[0])
        self.start = self.goal = None
        starts = goals = 0

        for r in range(self.rows):               # scan the grid to find S and G
            if len(self.grid[r]) != self.cols:
                raise ValueError(f"Row {r} has a different length")
            for c in range(self.cols):
                ch = self.grid[r][c]
                if ch == "S":
                    self.start = (r, c)
                    starts += 1
                elif ch == "G":
                    self.goal = (r, c)
                    goals += 1
                elif ch != "#" and ch not in COSTS:      # unknown symbol
                    raise ValueError(f"Unknown symbol {ch!r} at {(r, c)}")

        if starts != 1 or goals != 1:
            raise ValueError("Map needs exactly one S and one G")

    def is_goal(self, state):
        return state == self.goal

    def neighbors(self, state):
        r, c = state
        for dr, dc in ((-1, 0), (0, 1), (1, 0), (0, -1)):   # Up, Right, Down, Left
            nr, nc = r + dr, c + dc
            inside = 0 <= nr < self.rows and 0 <= nc < self.cols   # True only if the cell is on the map
            if inside and self.grid[nr][nc] != "#":                # wall check
                yield (nr, nc), COSTS[self.grid[nr][nc]]

    def manhattan(self, state):   # heuristic h(n): |row diff| + |col diff|
        return abs(state[0] - self.goal[0]) + abs(state[1] - self.goal[1])

In [41]:
def reconstruct(parent, state):  # Walk backward through the parent dict, then reverse
    path = []
    while state is not None:
        path.append(state)
        state = parent[state]
    return path[::-1]


def path_cost(problem, path): # Total cost of a path
    return sum(COSTS[problem.grid[r][c]] for r, c in path[1:])

## 2. Uninformed Algorithms

The first three algorithms are uninformed, which means they have no idea where the goal is. All they do is follow a simple rule to decide which cell to explore next. The list of cells that are waiting to be explored is called the frontier, and the only real difference between these algorithms is how they order it.

**Breadth-First Search (BFS)** explores like a ripple spreading across a pond. It first looks at every cell one step away from the start, then every cell two steps away, and so on. To do this it keeps the frontier in a queue, so the oldest cell is always explored first. BFS finds the path with the fewest moves, but it never looks at costs. For our robot this means it can choose a short path that crosses a lot of rough floor and ends up expensive, which is exactly what happens on MAP_B.

**Depth-First Search (DFS)** behaves more like someone walking through a maze, picking a direction and following it as far as it goes before backing up when it gets stuck. It keeps the frontier in a stack, so the newest cell is always explored first. DFS needs very little memory, but the path it returns depends heavily on the order in which it tries directions. It can easily return a long and expensive path, as it did on MAP_A with a cost of 31, so it cannot be trusted to find the cheapest route.

**Uniform-Cost Search (UCS)** is like BFS that actually cares about cost. It always extends whichever path is cheapest so far, using a priority queue ordered by the real cost paid to reach each cell. It also waits until the goal is picked from the queue before stopping, so a cheaper route discovered later is never missed. This makes UCS always find the cheapest path. Its weakness is that it spreads out in every direction, even away from the goal, so it often explores more cells than it needs to.

In [42]:
def bfs(problem):
    start = problem.start
    frontier = deque([start])       # queue (FIFO), faster than a list
    parent = {start: None}
    expanded = 0
    max_frontier = 1
    while frontier:
        state = frontier.popleft()
        expanded += 1
        if problem.is_goal(state):
            return reconstruct(parent, state), expanded, max_frontier
        for nxt, _ in problem.neighbors(state):
            if nxt not in parent:
                parent[nxt] = state
                frontier.append(nxt)
        max_frontier = max(max_frontier, len(frontier))   # track the biggest frontier
    return None, expanded, max_frontier

In [43]:
def dfs(problem):
    start = problem.start
    frontier = [start]              # stack (LIFO)
    parent = {start: None}
    visited = set()
    expanded = 0
    max_frontier = 1
    while frontier:
        state = frontier.pop()      # last in, first out
        if state in visited:
            continue
        visited.add(state)
        expanded += 1
        if problem.is_goal(state):
            return reconstruct(parent, state), expanded, max_frontier
        for nxt, _ in problem.neighbors(state):
            if nxt not in visited:
                parent[nxt] = state
                frontier.append(nxt)
        max_frontier = max(max_frontier, len(frontier))
    return None, expanded, max_frontier

In [44]:
def ucs(problem):                   # cares about the cost so far g(n)
    start = problem.start
    frontier = [(0, start)]         # priority queue of (g, state)
    parent = {start: None}
    best_g = {start: 0}
    closed = set()
    expanded = 0
    max_frontier = 1
    while frontier:
        g, state = heapq.heappop(frontier)
        if state in closed:         # stale entry (lazy deletion)
            continue
        closed.add(state)
        expanded += 1
        if problem.is_goal(state):
            return reconstruct(parent, state), expanded, max_frontier
        for nxt, step in problem.neighbors(state):
            new_g = g + step
            if nxt not in best_g or new_g < best_g[nxt]:   # new cell, or a cheaper way to it
                best_g[nxt] = new_g
                parent[nxt] = state
                heapq.heappush(frontier, (new_g, nxt))
        max_frontier = max(max_frontier, len(frontier))
    return None, expanded, max_frontier

## 3. Informed Algorithms and Heuristic

The next two algorithms are informed. Besides the cost of the cells, they use a heuristic, which is a quick estimate of how far each cell is from the goal. You can think of it as a compass that points the search toward G.

**Our heuristic: Manhattan distance.** For any cell, we add up the number of rows and the number of columns that separate it from the goal. Written as a formula, h(n) = |row of n − row of G| + |column of n − column of G|. This estimate is safe to use because the robot needs at least that many moves to reach G, and every move costs at least 1, so the estimate can never be higher than the true cost. A heuristic with this property is called admissible, and it is what allows A* to guarantee the cheapest path. It is also consistent, because a single move changes the estimate by at most 1 while costing at least 1.

**Greedy Best-First Search** always walks toward whichever cell looks closest to the goal, without thinking about how much the trip has already cost. Its priority queue is ordered by the heuristic alone. This makes it very fast and it explores very few cells, only 9 on MAP_B. The downside is that it ignores cost completely. On MAP_B it marches straight along the rough floor and pays 22, when the cheapest path costs only 12. It is fast, but it is not guaranteed to find the cheapest path.

**A* Search** combines the strengths of UCS and Greedy. For every cell it considers both what the trip has cost so far and how far the goal seems to be, adding them together as f(n) = g(n) + h(n), where g(n) is the real cost so far and h(n) is the estimated cost remaining. Its priority queue is ordered by this total. Because the heuristic is admissible, A* finds the same cheapest path as UCS, but it explores fewer cells since the heuristic keeps it pointed toward the goal. On MAP_B it expanded 15 cells where UCS needed 30.

In [45]:
def greedy(problem):                # cares about h(n) only
    start = problem.start
    frontier = [(problem.manhattan(start), start)]   # priority queue of (h, state)
    parent = {start: None}
    closed = set()
    expanded = 0
    max_frontier = 1
    while frontier:
        _, state = heapq.heappop(frontier)
        if state in closed:
            continue
        closed.add(state)
        expanded += 1
        if problem.is_goal(state):
            return reconstruct(parent, state), expanded, max_frontier
        for nxt, _ in problem.neighbors(state):
            if nxt not in parent:
                parent[nxt] = state
                heapq.heappush(frontier, (problem.manhattan(nxt), nxt))
        max_frontier = max(max_frontier, len(frontier))
    return None, expanded, max_frontier

In [46]:
def a_star(problem):                # cares about f(n) = g(n) + h(n)
    start = problem.start
    frontier = [(problem.manhattan(start), 0, start)]   # (f, g, state)
    parent = {start: None}
    best_g = {start: 0}
    closed = set()
    expanded = 0
    max_frontier = 1
    while frontier:
        _, g, state = heapq.heappop(frontier)
        if state in closed:
            continue
        closed.add(state)
        expanded += 1
        if problem.is_goal(state):
            return reconstruct(parent, state), expanded, max_frontier
        for nxt, step in problem.neighbors(state):
            new_g = g + step
            if nxt not in best_g or new_g < best_g[nxt]:
                best_g[nxt] = new_g
                parent[nxt] = state
                f = new_g + problem.manhattan(nxt)
                heapq.heappush(frontier, (f, new_g, nxt))
        max_frontier = max(max_frontier, len(frontier))
    return None, expanded, max_frontier

In [47]:
def draw(problem, path): # drawing the map for the output 
    on_path = set(path or [])
    lines = []
    for r in range(problem.rows):
        row = ""
        for c in range(problem.cols):
            ch = problem.grid[r][c]
            row += "*" if (r, c) in on_path and ch not in "SG" else ch
        lines.append(row)
    return "\n".join(lines)

In [ ]:
ALGORITHMS = [
    ("BFS",    bfs),
    ("DFS",    dfs),
    ("UCS",    ucs),
    ("Greedy", greedy),
    ("A*",     a_star),
]

def run_algorithm(fn, problem, repeats=100):
    path, expanded, max_frontier = fn(problem)
    t0 = time.perf_counter()
    for _ in range(repeats):
        fn(problem)
    ms = (time.perf_counter() - t0) * 1000 / repeats
    return path, expanded, max_frontier, ms

In [49]:
problem = GridProblem(GRID)

for name, fn in ALGORITHMS:
    path, expanded, max_frontier, ms = run_algorithm(fn, problem)
    print(f"\n===== {name} =====")
    if path is None:
        print("No path found")
        print("Nodes expanded:", expanded)
        continue
    print("Path:", path)                              # list of (row, column)
    print("Total cost:", path_cost(problem, path))
    print("Nodes expanded:", expanded)
    print(draw(problem, path))                        # path cells marked with *


===== BFS =====
Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (3, 7), (4, 7)]
Total cost: 15
Nodes expanded: 27
S**#....
##*#.##.
..******
.####.#*
...~..#G

===== DFS =====
Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 1), (2, 0), (3, 0), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4), (4, 5), (3, 5), (2, 5), (2, 4), (1, 4), (0, 4), (0, 5), (0, 6), (0, 7), (1, 7), (2, 7), (3, 7), (4, 7)]
Total cost: 31
Nodes expanded: 28
S**#****
##*#*##*
***.**.*
*####*#*
******#G

===== UCS =====
Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (3, 7), (4, 7)]
Total cost: 15
Nodes expanded: 28
S**#....
##*#.##.
..******
.####.#*
...~..#G

===== Greedy =====
Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (3, 7), (4, 7)]
Total cost: 15
Nodes expanded: 12
S**#....
##*#.##.
..******
.####.#*
...~..#G

===== A* =====
Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (2, 5), (2

In [50]:
maps = [("MAP_A", MAP_A), ("MAP_B", MAP_B), ("MAP_C", MAP_C)]

print(f"{'Map':<7}{'Algorithm':<10}{'Length':>8}{'Cost':>6}{'Expanded':>10}{'MaxFrontier':>13}{'Time (ms)':>11}")
print("-" * 65)

for map_name, grid in maps:
    problem = GridProblem(grid)
    for name, fn in ALGORITHMS:
        path, expanded, max_frontier, ms = run_algorithm(fn, problem)
        if path:
            print(f"{map_name:<7}{name:<10}{len(path)-1:>8}{path_cost(problem, path):>6}"
                  f"{expanded:>10}{max_frontier:>13}{ms:>11.3f}")
        else:
            print(f"{map_name:<7}{name:<10}{'No path found':>14}{expanded:>10}{max_frontier:>13}{ms:>11.3f}")
    print()

Map    Algorithm   Length  Cost  Expanded  MaxFrontier  Time (ms)
-----------------------------------------------------------------
MAP_A  BFS             11    15        27            5      0.029
MAP_A  DFS             25    31        28            4      0.035
MAP_A  UCS             11    15        28            4      0.034
MAP_A  Greedy          11    15        12            5      0.015
MAP_A  A*              11    15        21            5      0.027

MAP_B  BFS              8    22        23            4      0.021
MAP_B  DFS             16    16        27           10      0.026
MAP_B  UCS             12    12        30            4      0.041
MAP_B  Greedy           8    22         9            2      0.011
MAP_B  A*              12    12        15            7      0.024

MAP_C  BFS        No path found        13            3      0.022
MAP_C  DFS        No path found        13            3      0.014
MAP_C  UCS        No path found        13            3      0.016
MAP_C  G

In [51]:
# Path drawings for ALL maps (use these for the report screenshots)
for map_name, grid in maps:
    problem = GridProblem(grid)
    print(f"\n########## {map_name} ##########")
    for name, fn in ALGORITHMS:
        path, expanded, max_frontier, ms = run_algorithm(fn, problem, repeats=1)
        if path is None:
            print(f"\n{name}: No path found")
        else:
            print(f"\n{name}  (cost {path_cost(problem, path)}, expanded {expanded})")
            print(draw(problem, path))


########## MAP_A ##########

BFS  (cost 15, expanded 27)
S**#....
##*#.##.
..******
.####.#*
...~..#G

DFS  (cost 31, expanded 28)
S**#****
##*#*##*
***.**.*
*####*#*
******#G

UCS  (cost 15, expanded 28)
S**#....
##*#.##.
..******
.####.#*
...~..#G

Greedy  (cost 15, expanded 12)
S**#....
##*#.##.
..******
.####.#*
...~..#G

A*  (cost 15, expanded 21)
S**#....
##*#.##.
..******
.####.#*
...~..#G

########## MAP_B ##########

BFS  (cost 22, expanded 23)
S*******G
.#######.
.........
.#.#.#.#.
.........

DFS  (cost 16, expanded 27)
S~~~~~~~G
*#######*
*.......*
*#.#.#.#*
*********

UCS  (cost 12, expanded 30)
S~~~~~~~G
*#######*
*********
.#.#.#.#.
.........

Greedy  (cost 22, expanded 9)
S*******G
.#######.
.........
.#.#.#.#.
.........

A*  (cost 12, expanded 15)
S~~~~~~~G
*#######*
*********
.#.#.#.#.
.........

########## MAP_C ##########

BFS: No path found

DFS: No path found

UCS: No path found

Greedy: No path found

A*: No path found


## 4. Results

We ran all five algorithms on the three maps with the same costs and the same direction order. The results are below.

| Map | Algorithm | Path length | Cost | Nodes expanded | Max frontier | Time (ms) |
|---|---|---|---|---|---|---|
| MAP_A | BFS | 11 | 15 | 27 | 5 | 0.035 |
| MAP_A | DFS | 25 | 31 | 28 | 4 | 0.028 |
| MAP_A | UCS | 11 | 15 | 28 | 4 | 0.041 |
| MAP_A | Greedy | 11 | 15 | 12 | 5 | 0.015 |
| MAP_A | A* | 11 | 15 | 21 | 5 | 0.028 |
| MAP_B | BFS | 8 | 22 | 23 | 4 | 0.021 |
| MAP_B | DFS | 16 | 16 | 27 | 10 | 0.028 |
| MAP_B | UCS | 12 | 12 | 30 | 4 | 0.034 |
| MAP_B | Greedy | 8 | 22 | 9 | 2 | 0.009 |
| MAP_B | A* | 12 | 12 | 15 | 7 | 0.022 |
| MAP_C | BFS | No path found | | 13 | 3 | 0.011 |
| MAP_C | DFS | No path found | | 13 | 3 | 0.011 |
| MAP_C | UCS | No path found | | 13 | 3 | 0.017 |
| MAP_C | Greedy | No path found | | 13 | 2 | 0.016 |
| MAP_C | A* | No path found | | 13 | 3 | 0.016 |

A few words on how these numbers were measured. The path length is the number of moves, so the number of cells in the path minus one. The cost is the sum of the costs of the cells entered. The nodes expanded count includes the goal cell, since it is expanded when it is taken from the frontier. The maximum frontier is the largest size the frontier reached during the search; for UCS, Greedy and A* it can include outdated duplicate entries, because heapq cannot delete items from the middle of the queue. The time is the average of 100 runs, and since these maps are tiny the values are very small and change a little from one machine to another.

On MAP_A, BFS, UCS, Greedy and A* all found the best cost of 15, while DFS wandered around and ended up with 31. MAP_B is the trap map. Its top row is the shortest route at 8 steps, but it crosses seven rough cells and costs 22. BFS and Greedy walked straight into it. UCS and A* noticed the cost and took a longer 12-step detour that costs only 12. DFS landed on a cost of 16, but that was only luck from the order in which it tries directions, so it gives no guarantee. On MAP_C, the walls completely separate S from G, so every algorithm explored all 13 reachable cells and correctly reported that no path exists.

## 7. Conclusion

This task showed us that the shortest path is not always the cheapest one. BFS and Greedy care about the number of steps or the estimated distance, and they ignore the real cost of rough floor, so on MAP_B they both returned a cost of 22 instead of the best possible 12. UCS and A* always found the cheapest path, and A* did it with up to half as much exploration because its Manhattan heuristic is admissible and consistent. DFS turned out to be unreliable for cost, since its result depends on the order of directions, and it gave the worst path on MAP_A with a cost of 31.

We also confirmed that all five algorithms handle a map with no solution properly. On MAP_C they exhausted every reachable cell and reported "No path found" instead of looping or crashing.

Overall, A* is the best choice for this warehouse robot, because it combines the optimal cost of UCS with the efficiency of an informed search.

**GitHub:** (paste link here)